In [1]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.stats import mannwhitneyu
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings, os, time, gc
warnings.filterwarnings('ignore')

print("✅ All libraries loaded")
print("=" * 70)



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 130.7 MB/s eta 0:00:00
  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 85.1 GB total
✅ All libraries loaded


In [2]:
# =================================================================
# CELL 2: Google Drive Mount + Output Directory
# =================================================================
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
'''
V18-v2 Result 4.1 Supplementary Materials
Purpose:

Generate TOX Liver Myeloid dot plot across disease spectrum (Supplementary Figure)
Compare with TOX CD4+ T (Figure 4A) to show lineage-specific pattern
Format IT→IA comprehensive comparison as Supplementary Table
Key finding: TOX in Liver Myeloid is IT-specific (NL→IT ★↑1857%, IT→IA ★↓96%), while TOX in Liver CD4+ T is chronic-persistent (stays elevated through IA). IA Myeloid TOX (0.0033) falls BELOW NL level (≈0.0044) — complete suppression.

Prerequisite: Run V18_C8_IT_IA_Transition_Verification_v2.ipynb first (adata must be loaded in memory)
'''

In [3]:
DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
RESULTS_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2'

print('Loading h5ad (backed mode)...')
adata = sc.read_h5ad(DATA_PATH, backed='r')
print(f'Shape: {adata.shape}')
print(f'Columns: {list(adata.obs.columns[:20])}')

Loading h5ad (backed mode)...
Shape: (243000, 24452)
Columns: ['sample', 'tissue', 'Stage', 'IT_cluster_21', 'IT_cluster_23', 'IT_cluster_25', 'IT_nk_collapse', 'IT_IT_signature', 'GSM_ID', 'IT_score_v2', 'IT_score_v3', 'IT_score_v4', 'IT_signature_final', 'IT_like', 'PW_mTOR_signaling', 'PW_glycolysis', 'PW_oxidative_phosphorylation', 'PW_nk_cell_cytotoxicity', 'PW_il15_signaling', 'PW_b_cell_differentiation']


In [19]:
# Cell 1: Setup (skip if adata already loaded from verification notebook)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.size'] = 10

# If adata not loaded, mount and load
try:
    _ = adata.shape
    print(f'adata already loaded: {adata.shape}')
except:
    from google.colab import drive
    drive.mount('/content/drive')
    import scanpy as sc
    DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
    print('Loading h5ad...')
    adata = sc.read_h5ad(DATA_PATH)
    print(f'Loaded: {adata.shape}')

RESULTS_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2'
import os

# Ensure obs is available
obs = adata.obs.copy()

# Auto-detect columns (same logic as verification notebook)
COL_STAGE = 'Stage' if 'Stage' in obs.columns else [c for c in obs.columns if 'stage' in c.lower() or 'disease' in c.lower()][0]
COL_LINEAGE = 'major_lineage' if 'major_lineage' in obs.columns else [c for c in obs.columns if 'lineage' in c.lower() or 'cell_type' in c.lower()][0]

# Tissue
if 'tissue' in obs.columns:
    COL_TISSUE = 'tissue'
elif 'Tissue' in obs.columns:
    COL_TISSUE = 'Tissue'
else:
    for c in ['sample', 'Sample', 'orig.ident']:
        if c in obs.columns:
            obs['tissue_derived'] = obs[c].apply(
                lambda x: 'Liver' if ('Liver' in str(x) or '_L_' in str(x) or str(x).endswith('_L'))
                else ('Blood' if ('PBMC' in str(x) or '_P_' in str(x) or str(x).endswith('_P') or 'Blood' in str(x))
                else 'Unknown'))
            COL_TISSUE = 'tissue_derived'
            break

# Donor
COL_DONOR = None
for c in ['donor', 'Donor', 'patient', 'subject', 'donor_id']:
    if c in obs.columns:
        COL_DONOR = c
        break
if COL_DONOR is None:
    for c in ['sample', 'Sample', 'orig.ident']:
        if c in obs.columns:
            obs['donor_derived'] = obs[c].astype(str).str.split('_').str[1]
            COL_DONOR = 'donor_derived'
            break

print(f'Stage: {COL_STAGE} → {sorted(obs[COL_STAGE].unique())}')
print(f'Lineage: {COL_LINEAGE}')
print(f'Tissue: {COL_TISSUE} → {sorted(obs[COL_TISSUE].unique())}')
print(f'Donor: {COL_DONOR} → n={obs[COL_DONOR].nunique()}')

adata already loaded: (243000, 24452)
Stage: Stage → ['AR', 'CR', 'IA', 'IT', 'NL']
Lineage: major_lineage
Tissue: tissue → ['Blood', 'Liver']
Donor: donor_derived → n=23


In [20]:
# Cell 2: Extract donor-level expression for TOX across ALL disease groups
from scipy.stats import mannwhitneyu

def get_donor_expression(adata, obs_df, gene, lineage, tissue,
                         col_stage=COL_STAGE, col_lineage=COL_LINEAGE,
                         col_tissue=COL_TISSUE, col_donor=COL_DONOR):
    """
    Get donor-level mean expression for a gene in a specific lineage/tissue,
    across ALL disease groups.
    Returns DataFrame with columns: donor, stage, mean_expr
    """
    mask = (
        (obs_df[col_lineage] == lineage) &
        (obs_df[col_tissue] == tissue)
    )
    cells = obs_df[mask].copy()

    if len(cells) == 0:
        return pd.DataFrame()

    # Get gene expression
    gene_idx = list(adata.var_names).index(gene)
    cell_indices = np.where(mask.values)[0]

    try:
        chunk = adata.X[cell_indices, :]
        if hasattr(chunk, 'toarray'):
            expr = np.asarray(chunk[:, gene_idx].toarray()).flatten()
        else:
            expr = np.asarray(chunk[:, gene_idx]).flatten()
    except:
        col_data = adata[:, gene].X
        if hasattr(col_data, 'toarray'):
            col_data = col_data.toarray().flatten()
        else:
            col_data = np.asarray(col_data).flatten()
        expr = col_data[cell_indices]

    cells['expr'] = expr

    # Donor-level means
    donor_means = cells.groupby([col_donor, col_stage])['expr'].mean().reset_index()
    donor_means.columns = ['donor', 'stage', 'mean_expr']

    return donor_means


def donor_level_test(adata, obs_df, gene, lineage, tissue,
                     group_a='IT', group_b='IA',
                     col_stage=COL_STAGE, col_lineage=COL_LINEAGE,
                     col_tissue=COL_TISSUE, col_donor=COL_DONOR):
    """Donor-level Mann-Whitney U test between two groups."""
    mask = (
        (obs_df[col_stage].isin([group_a, group_b])) &
        (obs_df[col_lineage] == lineage) &
        (obs_df[col_tissue] == tissue)
    )
    cells = obs_df[mask].copy()
    if len(cells) == 0:
        return None
    gene_idx = list(adata.var_names).index(gene)
    cell_indices = np.where(mask.values)[0]
    try:
        chunk = adata.X[cell_indices, :]
        if hasattr(chunk, 'toarray'):
            expr = np.asarray(chunk[:, gene_idx].toarray()).flatten()
        else:
            expr = np.asarray(chunk[:, gene_idx]).flatten()
    except:
        col_data = adata[:, gene].X
        if hasattr(col_data, 'toarray'):
            col_data = col_data.toarray().flatten()
        else:
            col_data = np.asarray(col_data).flatten()
        expr = col_data[cell_indices]
    cells['expr'] = expr
    donor_means = cells.groupby([col_donor, col_stage])['expr'].mean().reset_index()
    vals_a = donor_means[donor_means[col_stage] == group_a]['expr'].dropna().values
    vals_b = donor_means[donor_means[col_stage] == group_b]['expr'].dropna().values
    n_a, n_b = len(vals_a), len(vals_b)
    if n_a < 2 or n_b < 2:
        return None
    mean_a, mean_b = float(np.mean(vals_a)), float(np.mean(vals_b))
    try:
        _, p_val = mannwhitneyu(vals_a, vals_b, alternative='two-sided')
        p_val = float(p_val)
    except:
        p_val = 1.0
    pct_change = (mean_b - mean_a) / mean_a * 100 if mean_a > 1e-10 else (float('inf') if mean_b > 1e-10 else 0.0)
    direction = 'up' if mean_b > mean_a else 'down'
    n_consistent = sum(1 for va in vals_a for vb in vals_b
                       if (direction == 'up' and vb > va) or (direction == 'down' and vb < va))
    return {
        'gene': gene, 'lineage': lineage, 'tissue': tissue,
        'comparison': f'{group_a}\u2192{group_b}',
        f'n_{group_a}': n_a, f'n_{group_b}': n_b,
        f'mean_{group_a}': round(mean_a, 4), f'mean_{group_b}': round(mean_b, 4),
        'pct_change': round(pct_change, 1),
        'direction': '\u2191' if mean_b > mean_a else '\u2193',
        'p_value': round(p_val, 4),
        'sig': '\u2605' if p_val < 0.05 else ('\u2020' if p_val < 0.10 else ''),
        'consistency': f'{n_consistent}/{n_a * n_b}',
    }


print('Functions defined: get_donor_expression, donor_level_test')

# Test
df_test = get_donor_expression(adata, obs, 'TOX', 'Myeloid', 'Liver')
print(f'TOX Liver Myeloid — donors per stage:')
print(df_test.groupby('stage')['mean_expr'].agg(['count', 'mean', 'std']).round(4))

Functions defined: get_donor_expression, donor_level_test
TOX Liver Myeloid — donors per stage:
       count    mean     std
stage                       
CR         3  0.0178  0.0146
AR         3  0.0175  0.0155
IA         5  0.0033  0.0074
IT         6  0.0860  0.1107
NL         6  0.0044  0.0069


In [21]:
# Cell 3: Generate dot plot — TOX Liver Myeloid vs CD4+ T (disease spectrum)

# V18 graph rules: Liver = red circle, Blood = blue triangle
# Disease groups: NL, IT, IA, AR, CR
# Donor-level dots with group means

STAGE_ORDER = ['NL', 'IT', 'IA', 'AR', 'CR']
STAGE_COLORS = {'NL': '#2ca02c', 'IT': '#d62728', 'IA': '#ff7f0e',
                'AR': '#1f77b4', 'CR': '#9467bd'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Panel A: TOX Liver Myeloid ---
ax = axes[0]
df_myeloid = get_donor_expression(adata, obs, 'TOX', 'Myeloid', 'Liver')

for i, stage in enumerate(STAGE_ORDER):
    subset = df_myeloid[df_myeloid['stage'] == stage]['mean_expr'].values
    if len(subset) == 0:
        continue
    # Jitter
    jitter = np.random.uniform(-0.15, 0.15, len(subset))
    ax.scatter([i] * len(subset) + jitter, subset,
              c='#d44e4e', marker='o', s=60, alpha=0.8, edgecolors='white', linewidth=0.5,
              zorder=3)
    # Mean bar
    mean_val = np.mean(subset)
    ax.plot([i - 0.25, i + 0.25], [mean_val, mean_val],
           color='#d44e4e', linewidth=2.5, zorder=4)

# Add significance annotations for NL→IT and IT→IA
# NL→IT: p=0.004
y_max = df_myeloid['mean_expr'].max() * 1.15
ax.annotate('', xy=(1, y_max * 0.90), xytext=(0, y_max * 0.90),
           arrowprops=dict(arrowstyle='-', color='red', lw=1))
ax.text(0.5, y_max * 0.92, '★p=0.004', ha='center', va='bottom',
        fontsize=8, color='red')

# IT→IA: p=0.007
ax.annotate('', xy=(2, y_max * 0.80), xytext=(1, y_max * 0.80),
           arrowprops=dict(arrowstyle='-', color='blue', lw=1))
ax.text(1.5, y_max * 0.82, '★p=0.007', ha='center', va='bottom',
        fontsize=8, color='blue')

ax.set_xticks(range(len(STAGE_ORDER)))
ax.set_xticklabels(STAGE_ORDER)
ax.set_ylabel('Expression', fontsize=11)
ax.set_title('A. TOX (Myeloid, Liver)\nIT-Specific Transition', fontsize=11, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# --- Panel B: TOX Liver CD4+ T ---
ax = axes[1]
df_cd4 = get_donor_expression(adata, obs, 'TOX', 'CD4_T', 'Liver')

for i, stage in enumerate(STAGE_ORDER):
    subset = df_cd4[df_cd4['stage'] == stage]['mean_expr'].values
    if len(subset) == 0:
        continue
    jitter = np.random.uniform(-0.15, 0.15, len(subset))
    ax.scatter([i] * len(subset) + jitter, subset,
              c='#d44e4e', marker='o', s=60, alpha=0.8, edgecolors='white', linewidth=0.5,
              zorder=3)
    mean_val = np.mean(subset)
    ax.plot([i - 0.25, i + 0.25], [mean_val, mean_val],
           color='#d44e4e', linewidth=2.5, zorder=4)

# NL→IT significance from Results 2.3: p=0.009
y_max2 = df_cd4['mean_expr'].max() * 1.15
ax.annotate('', xy=(1, y_max2 * 0.90), xytext=(0, y_max2 * 0.90),
           arrowprops=dict(arrowstyle='-', color='red', lw=1))
ax.text(0.5, y_max2 * 0.92, '★p=0.009', ha='center', va='bottom',
        fontsize=8, color='red')

# IT→IA: NS (add NS annotation)
ax.text(1.5, y_max2 * 0.82, 'IT→IA: NS', ha='center', va='bottom',
        fontsize=8, color='gray')

ax.set_xticks(range(len(STAGE_ORDER)))
ax.set_xticklabels(STAGE_ORDER)
ax.set_ylabel('Expression', fontsize=11)
ax.set_title('B. TOX (CD4+ T, Liver)\nChronic-Persistent', fontsize=11, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

# Save
fig_path = os.path.join(RESULTS_DIR, 'FigS_TOX_Myeloid_vs_CD4T_Liver.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor='white')
print(f'Saved: {fig_path}')
plt.show()

Saved: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/FigS_TOX_Myeloid_vs_CD4T_Liver.png


In [23]:
# ============================================================
# Cell 4: Print exact donor-level values for TOX Myeloid Liver
# Verify: IA level vs NL level
# Cell 4 REPLACEMENT — paste over existing Cell 4
# Fix: dropna() for each stage subset
# ============================================================

print('=== TOX Liver Myeloid — Donor-Level Values ===')
df_myeloid = get_donor_expression(adata, obs, 'TOX', 'Myeloid', 'Liver')

for stage in STAGE_ORDER:
    subset = df_myeloid[df_myeloid['stage'] == stage]['mean_expr'].dropna()
    vals = subset.values
    if len(vals) > 0:
        print(f'  {stage} (n={len(vals)}): mean={np.mean(vals):.4f}, '
              f'values={[round(float(v), 4) for v in sorted(vals)]}')
    else:
        print(f'  {stage}: no data')

# NL vs IA test
nl_vals = df_myeloid[df_myeloid['stage'] == 'NL']['mean_expr'].dropna().values
it_vals = df_myeloid[df_myeloid['stage'] == 'IT']['mean_expr'].dropna().values
ia_vals = df_myeloid[df_myeloid['stage'] == 'IA']['mean_expr'].dropna().values

print(f'\n  NL→IT: mean NL={np.mean(nl_vals):.4f} → IT={np.mean(it_vals):.4f}, '
      f'{(np.mean(it_vals)-np.mean(nl_vals))/np.mean(nl_vals)*100:+.1f}%')
if len(nl_vals) >= 2 and len(it_vals) >= 2:
    _, p = mannwhitneyu(nl_vals, it_vals, alternative='two-sided')
    print(f'         p={p:.4f}')

if len(nl_vals) >= 2 and len(ia_vals) >= 2:
    _, p = mannwhitneyu(nl_vals, ia_vals, alternative='two-sided')
    pct = (np.mean(ia_vals) - np.mean(nl_vals)) / np.mean(nl_vals) * 100
    print(f'\n  NL→IA: mean NL={np.mean(nl_vals):.4f} → IA={np.mean(ia_vals):.4f}, '
          f'{pct:+.1f}%, p={p:.4f}')
    print(f'  → IA is {"BELOW" if np.mean(ia_vals) < np.mean(nl_vals) else "ABOVE"} NL level')

if len(it_vals) >= 2 and len(ia_vals) >= 2:
    _, p = mannwhitneyu(it_vals, ia_vals, alternative='two-sided')
    pct = (np.mean(ia_vals) - np.mean(it_vals)) / np.mean(it_vals) * 100
    print(f'\n  IT→IA: mean IT={np.mean(it_vals):.4f} → IA={np.mean(ia_vals):.4f}, '
          f'{pct:+.1f}%, p={p:.4f}')

print()
print('=== TOX Liver CD4+ T — Donor-Level Values ===')
df_cd4 = get_donor_expression(adata, obs, 'TOX', 'CD4_T', 'Liver')

for stage in STAGE_ORDER:
    subset = df_cd4[df_cd4['stage'] == stage]['mean_expr'].dropna()
    vals = subset.values
    if len(vals) > 0:
        print(f'  {stage} (n={len(vals)}): mean={np.mean(vals):.4f}, '
              f'values={[round(float(v), 4) for v in sorted(vals)]}')

nl_cd4 = df_cd4[df_cd4['stage'] == 'NL']['mean_expr'].dropna().values
it_cd4 = df_cd4[df_cd4['stage'] == 'IT']['mean_expr'].dropna().values
ia_cd4 = df_cd4[df_cd4['stage'] == 'IA']['mean_expr'].dropna().values

if len(nl_cd4) >= 2 and len(it_cd4) >= 2:
    _, p = mannwhitneyu(nl_cd4, it_cd4, alternative='two-sided')
    print(f'\n  NL→IT: mean NL={np.mean(nl_cd4):.4f} → IT={np.mean(it_cd4):.4f}, p={p:.4f}')

if len(it_cd4) >= 2 and len(ia_cd4) >= 2:
    _, p = mannwhitneyu(it_cd4, ia_cd4, alternative='two-sided')
    print(f'  IT→IA: mean IT={np.mean(it_cd4):.4f} → IA={np.mean(ia_cd4):.4f}, p={p:.4f}')

if len(nl_cd4) >= 2 and len(ia_cd4) >= 2:
    _, p = mannwhitneyu(nl_cd4, ia_cd4, alternative='two-sided')
    print(f'  NL→IA: mean NL={np.mean(nl_cd4):.4f} → IA={np.mean(ia_cd4):.4f}, p={p:.4f}')

# Summary comparison
print('\n' + '='*70)
print('TOX LINEAGE-SPECIFIC PATTERN SUMMARY (Liver)')
print('='*70)
print(f'{"":>10s} | {"NL":>8s} | {"IT":>8s} | {"IA":>8s} | {"NL→IT":>10s} | {"IT→IA":>10s}')
print('-'*70)
print(f'{"Myeloid":>10s} | {np.mean(nl_vals):.4f} | {np.mean(it_vals):.4f} | '
      f'{np.mean(ia_vals):.4f} | ★↑{(np.mean(it_vals)-np.mean(nl_vals))/np.mean(nl_vals)*100:.0f}% | '
      f'★↓{abs((np.mean(ia_vals)-np.mean(it_vals))/np.mean(it_vals)*100):.0f}%')
print(f'{"CD4+ T":>10s} | {np.mean(nl_cd4):.4f} | {np.mean(it_cd4):.4f} | '
      f'{np.mean(ia_cd4):.4f} | ★↑{(np.mean(it_cd4)-np.mean(nl_cd4))/np.mean(nl_cd4)*100:.0f}% | '
      f'NS')
print()
print(f'Myeloid: IA ({np.mean(ia_vals):.4f}) vs NL ({np.mean(nl_vals):.4f}) → '
      f'{"IA < NL (overcorrection)" if np.mean(ia_vals) < np.mean(nl_vals) else "IA ≈ NL (normalization)"}')
print(f'CD4+ T:  IA ({np.mean(ia_cd4):.4f}) vs NL ({np.mean(nl_cd4):.4f}) → '
      f'IA remains elevated (chronic-persistent)')

=== TOX Liver Myeloid — Donor-Level Values ===
  NL (n=6): mean=0.0044, values=[0.0, 0.0, 0.0, 0.0, 0.0114, 0.015]
  IT (n=6): mean=0.0860, values=[0.0182, 0.0241, 0.0427, 0.0553, 0.0666, 0.3088]
  IA (n=5): mean=0.0033, values=[0.0, 0.0, 0.0, 0.0, 0.0166]
  AR (n=3): mean=0.0175, values=[0.0, 0.023, 0.0296]
  CR (n=3): mean=0.0178, values=[0.0078, 0.0111, 0.0345]

  NL→IT: mean NL=0.0044 → IT=0.0860, +1856.8%
         p=0.0043

  NL→IA: mean NL=0.0044 → IA=0.0033, -24.6%, p=0.9076
  → IA is BELOW NL level

  IT→IA: mean IT=0.0860 → IA=0.0033, -96.1%, p=0.0067

=== TOX Liver CD4+ T — Donor-Level Values ===
  NL (n=6): mean=0.1577, values=[0.077, 0.1266, 0.1419, 0.1832, 0.1924, 0.225]
  IT (n=6): mean=0.3164, values=[0.193, 0.2094, 0.2419, 0.2423, 0.4808, 0.5311]
  IA (n=5): mean=0.2683, values=[0.0701, 0.273, 0.3085, 0.3434, 0.3468]
  AR (n=3): mean=0.2634, values=[0.2265, 0.237, 0.3266]
  CR (n=3): mean=0.2251, values=[0.21, 0.2128, 0.2526]

  NL→IT: mean NL=0.1577 → IT=0.3164, p=0.00

In [24]:
# Cell 5: Also generate GZMK Liver Myeloid dot plot for completeness

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for panel_idx, (gene, lineage, title_suffix) in enumerate([
    ('GZMK', 'Myeloid', 'IT-Specific Transition'),
    ('GZMK', 'CD8_T', 'IT→IA Downregulation')
]):
    ax = axes[panel_idx]
    df_plot = get_donor_expression(adata, obs, gene, lineage, 'Liver')

    for i, stage in enumerate(STAGE_ORDER):
        subset = df_plot[df_plot['stage'] == stage]['mean_expr'].values
        if len(subset) == 0:
            continue
        jitter = np.random.uniform(-0.15, 0.15, len(subset))
        ax.scatter([i] * len(subset) + jitter, subset,
                  c='#d44e4e', marker='o', s=60, alpha=0.8,
                  edgecolors='white', linewidth=0.5, zorder=3)
        mean_val = np.mean(subset)
        ax.plot([i - 0.25, i + 0.25], [mean_val, mean_val],
               color='#d44e4e', linewidth=2.5, zorder=4)

    ax.set_xticks(range(len(STAGE_ORDER)))
    ax.set_xticklabels(STAGE_ORDER)
    ax.set_ylabel('Expression', fontsize=11)
    label = 'A' if panel_idx == 0 else 'B'
    ax.set_title(f'{label}. GZMK ({lineage}, Liver)\n{title_suffix}',
                fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
fig_path2 = os.path.join(RESULTS_DIR, 'FigS_GZMK_Myeloid_vs_CD8T_Liver.png')
plt.savefig(fig_path2, dpi=300, bbox_inches='tight', facecolor='white')
print(f'Saved: {fig_path2}')
plt.show()

Saved: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/FigS_GZMK_Myeloid_vs_CD8T_Liver.png


In [25]:
# Cell 6: Blood NK transition genes — TGFB1, SERPINE1, ID3 dot plots

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

nk_genes = [
    ('TGFB1', '★p=0.016'),
    ('SERPINE1', '★p=0.016'),
    ('ID3', '★p=0.032'),
]

for panel_idx, (gene, it_ia_p_label) in enumerate(nk_genes):
    ax = axes[panel_idx]
    df_plot = get_donor_expression(adata, obs, gene, 'NK', 'Blood')

    for i, stage in enumerate(STAGE_ORDER):
        subset = df_plot[df_plot['stage'] == stage]['mean_expr'].values
        if len(subset) == 0:
            continue
        jitter = np.random.uniform(-0.12, 0.12, len(subset))
        ax.scatter([i] * len(subset) + jitter, subset,
                  c='#4878a8', marker='^', s=60, alpha=0.8,
                  edgecolors='white', linewidth=0.5, zorder=3)
        mean_val = np.mean(subset)
        ax.plot([i - 0.25, i + 0.25], [mean_val, mean_val],
               color='#4878a8', linewidth=2.5, zorder=4)

    # IT→IA annotation
    y_max = df_plot['mean_expr'].max() * 1.15
    ax.annotate('', xy=(2, y_max * 0.85), xytext=(1, y_max * 0.85),
               arrowprops=dict(arrowstyle='-', color='blue', lw=1))
    ax.text(1.5, y_max * 0.87, it_ia_p_label, ha='center', va='bottom',
            fontsize=8, color='blue')

    ax.set_xticks(range(len(STAGE_ORDER)))
    ax.set_xticklabels(STAGE_ORDER)
    ax.set_ylabel('Expression', fontsize=11)
    label = chr(65 + panel_idx)  # A, B, C
    ax.set_title(f'{label}. {gene} (NK, Blood)\nIA-Emergent',
                fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
fig_path3 = os.path.join(RESULTS_DIR, 'FigS_BloodNK_IT_IA_transition.png')
plt.savefig(fig_path3, dpi=300, bbox_inches='tight', facecolor='white')
print(f'Saved: {fig_path3}')
plt.show()

Saved: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/FigS_BloodNK_IT_IA_transition.png


In [28]:
# ============================================================
# Cell 7: Generate Supplementary Table — IT→IA Comprehensive Comparison
# Includes NL→IT context for all tested gene-lineage-tissue combinations
# Cell 7 REPLACEMENT — Optimized: pre-extract gene expressions
# Reduces 576 h5ad accesses → 24 (one per gene)
# Expected time: 2-3 minutes instead of 15-30 minutes
# ============================================================

import time
t0 = time.time()

broad_genes = [
    'TGFB1', 'ID3', 'SERPINE1', 'TOX', 'GZMK', 'SNAI1',
    'SOCS1', 'SOCS3', 'LAYN', 'TOX2',
    'LGALS9', 'AIM2', 'MEFV', 'DNMT1', 'DNMT3A', 'TET2',
    'JAK1', 'MTOR', 'PRDM1',
    'FOXP3', 'GNLY', 'TIGIT', 'CTLA4', 'BCL6',
]
broad_genes = [g for g in broad_genes if g in adata.var_names]
print(f'Genes to test: {len(broad_genes)}')

lineages = sorted([l for l in obs[COL_LINEAGE].unique()
                    if l in ['NK', 'Myeloid', 'CD4_T', 'CD8_T']])
tissues = sorted(obs[COL_TISSUE].unique())
comparisons = [('NL', 'IT'), ('IT', 'IA'), ('NL', 'IA')]

# ====== STEP 1: Pre-extract all gene expressions (one h5ad access per gene) ======
print(f'\nStep 1: Pre-extracting {len(broad_genes)} genes...')
gene_expr = {}
for gi, gene in enumerate(broad_genes):
    try:
        col = adata[:, gene].X
        if hasattr(col, 'toarray'):
            col = col.toarray().flatten()
        else:
            col = np.asarray(col).flatten()
        gene_expr[gene] = col
        if (gi + 1) % 10 == 0:
            print(f'  {gi+1}/{len(broad_genes)} extracted...')
    except Exception as e:
        print(f'  ⚠️ {gene}: {e}')

t1 = time.time()
print(f'Pre-extraction done in {t1-t0:.1f}s')

# ====== STEP 2: Build donor-level means table (all in pandas, no h5ad access) ======
print(f'\nStep 2: Computing donor-level means...')

def fast_donor_test(gene, lineage, tissue, group_a, group_b):
    """Fast version using pre-extracted expression."""
    if gene not in gene_expr:
        return None

    mask = (
        (obs[COL_STAGE].isin([group_a, group_b])) &
        (obs[COL_LINEAGE] == lineage) &
        (obs[COL_TISSUE] == tissue)
    )
    if mask.sum() == 0:
        return None

    cell_indices = np.where(mask.values)[0]
    expr = gene_expr[gene][cell_indices]

    temp = obs.loc[mask, [COL_DONOR, COL_STAGE]].copy()
    temp['expr'] = expr

    donor_means = temp.groupby([COL_DONOR, COL_STAGE])['expr'].mean().reset_index()
    vals_a = donor_means[donor_means[COL_STAGE] == group_a]['expr'].dropna().values
    vals_b = donor_means[donor_means[COL_STAGE] == group_b]['expr'].dropna().values

    n_a, n_b = len(vals_a), len(vals_b)
    if n_a < 2 or n_b < 2:
        return None

    mean_a, mean_b = float(np.mean(vals_a)), float(np.mean(vals_b))
    try:
        _, p_val = mannwhitneyu(vals_a, vals_b, alternative='two-sided')
        p_val = float(p_val)
    except:
        p_val = 1.0

    if mean_a > 1e-10:
        pct = (mean_b - mean_a) / mean_a * 100
    else:
        pct = float('inf') if mean_b > 1e-10 else 0.0

    direction = '↑' if mean_b > mean_a else '↓'
    n_consistent = sum(1 for va in vals_a for vb in vals_b
                       if (mean_b > mean_a and vb > va) or (mean_b <= mean_a and vb < va))

    return {
        'p_value': round(p_val, 4),
        'pct_change': round(pct, 1),
        'direction': direction,
        'sig': '★' if p_val < 0.05 else ('†' if p_val < 0.10 else ''),
        'consistency': f'{n_consistent}/{n_a * n_b}',
        'n_a': n_a, 'n_b': n_b,
        'mean_a': round(mean_a, 4), 'mean_b': round(mean_b, 4),
    }

# ====== STEP 3: Run all tests ======
supp_rows = []
total_tests = len(broad_genes) * len(lineages) * len(tissues)
done = 0

for gene in broad_genes:
    for lin in lineages:
        for tis in tissues:
            row = {'Gene': gene, 'Lineage': lin, 'Tissue': tis}
            any_sig = False

            for group_a, group_b in comparisons:
                prefix = f'{group_a}→{group_b}'
                res = fast_donor_test(gene, lin, tis, group_a, group_b)
                if res:
                    row[f'{prefix}_change'] = f'{res["direction"]}{abs(res["pct_change"]):.1f}%'
                    row[f'{prefix}_p'] = res['p_value']
                    row[f'{prefix}_sig'] = res['sig']
                    row[f'{prefix}_consistency'] = res['consistency']
                    if res['p_value'] < 0.05:
                        any_sig = True
                else:
                    row[f'{prefix}_change'] = 'ND'
                    row[f'{prefix}_p'] = None
                    row[f'{prefix}_sig'] = ''
                    row[f'{prefix}_consistency'] = ''

            # Classify pattern
            nl_it_sig = row.get('NL→IT_p') and row['NL→IT_p'] < 0.05
            it_ia_sig = row.get('IT→IA_p') and row['IT→IA_p'] < 0.05
            nl_ia_sig = row.get('NL→IA_p') and row['NL→IA_p'] < 0.05

            if nl_it_sig and it_ia_sig:
                row['Pattern'] = 'TRANSITION'
            elif nl_it_sig and not it_ia_sig:
                row['Pattern'] = 'IT-persistent'
            elif not nl_it_sig and it_ia_sig:
                row['Pattern'] = 'IA-emergent'
            elif nl_ia_sig:
                row['Pattern'] = 'Chronic (NL→IA sig)'
            else:
                row['Pattern'] = 'NS'

            if any_sig or row['Pattern'] != 'NS':
                supp_rows.append(row)

            done += 1

t2 = time.time()

df_supp = pd.DataFrame(supp_rows)
print(f'\nDone: {done} combos tested in {t2-t1:.1f}s (total {t2-t0:.1f}s)')
print(f'Supplementary table: {len(df_supp)} rows with at least one significant comparison')
print(f'\nPattern distribution:')
print(df_supp['Pattern'].value_counts())

# Save
supp_path = os.path.join(RESULTS_DIR, 'SuppTable_IT_IA_Comprehensive_Comparison.csv')
df_supp.to_csv(supp_path, index=False)
print(f'\nSaved: {supp_path}')

Genes to test: 24

Step 1: Pre-extracting 24 genes...
  10/24 extracted...
  20/24 extracted...
Pre-extraction done in 58.0s

Step 2: Computing donor-level means...

Done: 192 combos tested in 9.5s (total 67.5s)
Supplementary table: 65 rows with at least one significant comparison

Pattern distribution:
Pattern
IT-persistent          41
Chronic (NL→IA sig)    17
IA-emergent             6
TRANSITION              1
Name: count, dtype: int64

Saved: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/SuppTable_IT_IA_Comprehensive_Comparison.csv


In [29]:
# Cell 8: Display formatted supplementary table

# Show IT→IA significant results first, sorted by p-value
it_ia_sig = df_supp[df_supp['IT→IA_sig'].isin(['★', '†'])].sort_values('IT→IA_p')

print('='*100)
print('SUPPLEMENTARY TABLE: IT→IA Significant Changes (with NL→IT context)')
print('='*100)
print(f'{"Gene":>10s} | {"Lineage":>8s} | {"Tissue":>6s} | '
      f'{"NL→IT":>15s} | {"IT→IA":>15s} | {"NL→IA":>15s} | {"Pattern":>15s}')
print('-'*100)

for _, row in it_ia_sig.iterrows():
    nl_it = f'{row.get("NL→IT_sig","")}{row.get("NL→IT_change","ND")} p={row.get("NL→IT_p","ND")}'
    it_ia = f'{row.get("IT→IA_sig","")}{row.get("IT→IA_change","ND")} p={row.get("IT→IA_p","ND")}'
    nl_ia = f'{row.get("NL→IA_sig","")}{row.get("NL→IA_change","ND")} p={row.get("NL→IA_p","ND")}'
    print(f'{row["Gene"]:>10s} | {row["Lineage"]:>8s} | {row["Tissue"]:>6s} | '
          f'{nl_it:>15s} | {it_ia:>15s} | {nl_ia:>15s} | {row["Pattern"]:>15s}')

print()
print('--- TRANSITION patterns (NL→IT sig AND IT→IA sig) ---')
trans = df_supp[df_supp['Pattern'] == 'TRANSITION']
for _, row in trans.iterrows():
    print(f'  {row["Gene"]} {row["Lineage"]} {row["Tissue"]}: '
          f'NL→IT {row["NL→IT_change"]} (p={row["NL→IT_p"]}), '
          f'IT→IA {row["IT→IA_change"]} (p={row["IT→IA_p"]})')

SUPPLEMENTARY TABLE: IT→IA Significant Changes (with NL→IT context)
      Gene |  Lineage | Tissue |           NL→IT |           IT→IA |           NL→IA |         Pattern
----------------------------------------------------------------------------------------------------
       TOX |  Myeloid |  Liver | ★↑1856.8% p=0.0043 | ★↓96.1% p=0.0067 | ↓24.6% p=0.9076 |      TRANSITION
      GZMK |  Myeloid |  Liver | ↑248.3% p=0.2946 | ★↓98.4% p=0.0116 | ↓94.3% p=0.2448 |     IA-emergent
     TGFB1 |       NK |  Blood | ↑22.8% p=0.3095 | ★↑43.9% p=0.0159 | ★↑76.6% p=0.0159 |     IA-emergent
  SERPINE1 |       NK |  Blood | ↑88.1% p=0.2045 | ★↑341.4% p=0.0159 | †↑730.2% p=0.0617 |     IA-emergent
      GZMK |    CD8_T |  Liver | †↑28.7% p=0.0649 | ★↓22.7% p=0.0303 |     ↓0.5% p=1.0 |     IA-emergent
       ID3 |       NK |  Blood | ↑111.2% p=0.2903 | ★↑770.3% p=0.0317 | ★↑1738.1% p=0.0179 |     IA-emergent
  SERPINE1 |    CD4_T |  Liver | ↓84.8% p=0.3261 | ★↑534.1% p=0.0444 |     ↓3.9% p=1.0 |  

In [30]:
# Cell 9: Summary for manuscript

print('='*70)
print('MANUSCRIPT REVISION GUIDE — Result 4.1')
print('='*70)
print()
print('SUPPLEMENTARY FIGURE: TOX lineage-specific pattern')
print(f'  File: {fig_path}')
print('  Panel A: TOX Liver Myeloid — IT-specific spike, IA returns below NL')
print('  Panel B: TOX Liver CD4+ T — chronic-persistent (stays elevated)')
print('  → Demonstrates lineage-specific transition behavior')
print()
print('SUPPLEMENTARY FIGURE: GZMK Liver patterns')
print(f'  File: {fig_path2}')
print('  Panel A: GZMK Liver Myeloid — aberrant IT→IA elimination')
print('  Panel B: GZMK Liver CD8+ T — IT→IA ★↓22.7%')
print()
print('SUPPLEMENTARY FIGURE: Blood NK IA-emergent')
print(f'  File: {fig_path3}')
print('  TGFB1, SERPINE1, ID3 in Blood NK — all IA-emergent')
print()
print('SUPPLEMENTARY TABLE: IT→IA Comprehensive Comparison')
print(f'  File: {supp_path}')
print(f'  Rows: {len(df_supp)}')
print(f'  Patterns: {dict(df_supp["Pattern"].value_counts())}')
print()
print('KEY FINDING: TOX Liver Myeloid is the ONLY true TRANSITION gene')
print('  — IT-specific aberrant expression completely eliminated at IA')
print('  — IA level falls BELOW NL (overcorrection?)')
print('  — Contrasts with TOX CD4+ T which is chronic-persistent')
print()
print('✅ All supplementary materials generated.')

MANUSCRIPT REVISION GUIDE — Result 4.1

SUPPLEMENTARY FIGURE: TOX lineage-specific pattern
  File: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/FigS_TOX_Myeloid_vs_CD4T_Liver.png
  Panel A: TOX Liver Myeloid — IT-specific spike, IA returns below NL
  Panel B: TOX Liver CD4+ T — chronic-persistent (stays elevated)
  → Demonstrates lineage-specific transition behavior

SUPPLEMENTARY FIGURE: GZMK Liver patterns
  File: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/FigS_GZMK_Myeloid_vs_CD8T_Liver.png
  Panel A: GZMK Liver Myeloid — aberrant IT→IA elimination
  Panel B: GZMK Liver CD8+ T — IT→IA ★↓22.7%

SUPPLEMENTARY FIGURE: Blood NK IA-emergent
  File: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/FigS_BloodNK_IT_IA_transition.png
  TGFB1, SERPINE1, ID3 in Blood NK — all IA-emergent

SUPPLEMENTARY TABLE: IT→IA Comprehensive Comparison
  File: /content/drive/MyDrive/ITLAS/results/version18-analysis-v2/SuppTable_IT_IA_Comprehensive_Comparison.csv
  Ro